<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Train_ALIGNNFF_Mlearn.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Train machine learning force-field with [ALIGNN](https://github.com/atomgptlab/alignn) for example Silicon dataset

# Table of contents

1. Install [ALIGNN](https://github.com/usnistgov/alignn)
2. Train a simple Silicon ALIGNN-FF,
3. Post-raining analysis, equation of state
4. ALIGNN-FF Ase Calculator: predict energy/forces, optimized strcture, EV curves and phonons
5. Run example MD (Melt-quench) to make amorphous Silicon
6. Phonon bandstructure for silicon
7. Scaling with number of atoms
8. Add ALIGNN-results to the JARVIS-Leaderboard

Author: Kamal Choudhary (drkamal@jhu.edu)

Install JARVIS-Leaderboard and ALIGNN

In [ ]:
%%time
import os
# !pip install -q alignn
# !pip uninstall dgl -y
os.chdir('/content')
# Clone ALIGNN repo to get example folder
if not os.path.exists('alignn'):
   !git clone https://github.com/atomgptlab/alignn.git
os.chdir("alignn")

!git checkout develop
# !pip install -q --no-cache-dir dgl -f https://data.dgl.ai/wheels/torch-2.4/cu124/repo.html
!pip install -q -e .
# !uv pip install -q -e .
os.chdir('/content')

In [ ]:
!nvcc --version

Make sure torch cuda version is installed properly

In [ ]:
import torch
torch.cuda.is_available()

In [ ]:
%%time
import os
os.chdir('/content')

!rm -rf jarvis_leaderboard
# Clone JARVIS-Leaderboard repo to get example folder
if not os.path.exists('jarvis_leaderboard'):
  !git clone https://github.com/atomgptlab/jarvis_leaderboard.git
os.chdir('jarvis_leaderboard')
!pip install -q -e .
os.chdir('/content')

## Runtime>> Restart session

In [ ]:
import alignn

In [ ]:
import requests

url = "https://ndownloader.figshare.com/files/46761919"
r = requests.get(url, allow_redirects=True)
open("mlearn_si_id_prop.json.zip", "wb").write(r.content)


In [ ]:
!ls

In [ ]:
!pwd

Lets look at default alignn-ff model config, Restart session

In [ ]:
import numpy
numpy.__version__

In [ ]:
from alignn.ff.ff import default_path, AlignnAtomwiseCalculator
path = default_path()
print(path)

In [ ]:
# !ls /usr/local/lib/python3.11/site-packages/alignn/ff/v12.2.2024_dft_3d_307k

In [ ]:
from jarvis.db.jsonutils import loadjson,dumpjson
import pprint
config=loadjson(path+'/config.json')
#pprint.pprint(config)

We are going to update n_train, n_val, n_test as per this dataset keeping evrything else fixed. These number are taken from jarvis-leaderboard


In [ ]:
config['n_train']=214
config['n_val']=25
config['n_test']=25
config['cutoff']=5
config['model']['name']="alignn_atomwise_pure" #Not using DGL
config['neighbor_strategy']="pure_torch" #Not using DGL
config['epochs']=150

# config['compute_line_graph']=False
# config["model"]['alignn_layers']=0


dumpjson(data=config, filename='config_si_mlearn.json')



In [ ]:
!unzip mlearn_si_id_prop.json.zip

In [ ]:
import os
from jarvis.db.jsonutils import loadjson,dumpjson
!rm mlearn_si_id_prop.json
if not os.path.exists('DataDir'):
  os.makedirs('DataDir')
!unzip mlearn_si_id_prop.json.zip
!mv mlearn_si_id_prop.json DataDir/id_prop.json


Check input data format

The DataDir/id_prop.json contains arrays of dictionaries. Each dictionary has keys such as 'id', 'atoms', 'forces', 'total_energy'.


An example to convert a vasprun.xml to id_prop.json is available [here](https://gist.github.com/knc6/5513b21f5fd83a7943509ffdf5c3608b).

In [ ]:
!ls

In [ ]:
from jarvis.db.jsonutils import loadjson
import pprint

d=loadjson('DataDir/id_prop.json')
# pprint.pprint(d[0])

In [ ]:
!rm -rf sampletrain_data  sampleval_data sampletest_data

In [ ]:
import alignn
alignn.__file__

In [ ]:
import torch
torch.__version__

In [ ]:
# # Only if you are using GPU based training
# !pip uninstall dgl -y
# !pip install --no-cache-dir dgl -f https://data.dgl.ai/wheels/torch-2.2/cu121/repo.html

In [ ]:

%%time
!train_alignn.py --root_dir DataDir/ --config config_si_mlearn.json --output_dir OutputDir

In [ ]:
!ls OutputDir

In [ ]:
# Example of loading pytorch model, not necessary if you are going to use ASE calculator later
from alignn.models.alignn_atomwise_pure import ALIGNNAtomWisePureConfig , ALIGNNAtomWisePure
import torch
from jarvis.db.jsonutils import loadjson
config=loadjson('OutputDir/config.json')
restart_model_path = 'OutputDir/best_model.pt'
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config['model']['stresswise_weight'] = 0.01 #to enable stress calculation
tmp = ALIGNNAtomWisePureConfig(**config["model"])
model = ALIGNNAtomWisePure(tmp)
model.load_state_dict(torch.load(restart_model_path, map_location=device))
model = model.to(device)

In [ ]:
# !wget https://figshare.com/ndownloader/files/40357663 -O mlearn.json.zip

In [ ]:
from jarvis.core.atoms import Atoms
from alignn.ff.ff import AlignnAtomwiseCalculator, default_path, ev_curve
import pandas as pd
import numpy as np
import zipfile
import json
import os
import glob
from alignn.graphs import Graph
import matplotlib.pyplot as plt
from ase.stress import full_3x3_to_voigt_6_stress, voigt_6_to_full_3x3_stress
from alignn.graphs import radius_graph_jarvis
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
calc = AlignnAtomwiseCalculator(path='OutputDir',device=device, force_mult_batchsize=False)


def get_pred(atoms=None):
    atoms=atoms.ase_converter()
    atoms.calc=calc
    return atoms.get_potential_energy(),atoms.get_forces(),atoms.get_stress()

    # # g,lg=radius_graph_jarvis(atoms=atoms,cutoff=config['cutoff'])
    # g,lg=Graph.atom_dgl_multigraph(atoms=atoms,cutoff=config['cutoff'])
    # out = model([g.to(device),lg.to(device)])
    # #print('out',out,out.keys())
    # energy = out['out'].cpu().detach().numpy()*atoms.num_atoms
    # forces = out['grad'].cpu().detach().numpy()*atoms.num_atoms
    # stress = np.zeros((3,3)) #out["stresses"][:3].reshape(3, 3).detach().cpu().numpy()
    # return energy,forces,stress






df = pd.DataFrame(
    json.loads(
        zipfile.ZipFile("mlearn_si_id_prop.json.zip").read(
            "mlearn_si_id_prop.json"
        )
    )
)
print(df)
element="Si"
# print(df)
for i in glob.glob("/content/jarvis_leaderboard/jarvis_leaderboard/benchmarks/AI/MLFF/*energy*.zip"):
    if "mlearn" in i and element in i:
        fname_e = (
            "AI-MLFF-energy-"
            + i.split("/")[-1].split("_energy.json.zip")[0]
            + "-test-mae.csv"
        )
        fname_f = (
            "AI-MLFF-forces-"
            + i.split("/")[-1].split("_energy.json.zip")[0]
            + "-test-multimae.csv"
        )
        fname_s = (
            "AI-MLFF-stresses-"
            + i.split("/")[-1].split("_energy.json.zip")[0]
            + "-test-multimae.csv"
        )
        f_e = open(fname_e, "w")
        f_f = open(fname_f, "w")
        f_s = open(fname_s, "w")

        f_e.write("id,target,prediction\n")
        f_f.write("id,target,prediction\n")
        f_s.write("id,target,prediction\n")
        #voigt_6_to_full_3x3_stress
        print(i)
        dat = json.loads(
            zipfile.ZipFile(i).read(i.split("/")[-1].split(".zip")[0])
        )
        print(dat["test"])
        for key, val in dat["test"].items():
            entry = df[df["jid"] == key]
            atoms = Atoms.from_dict(entry.atoms.values[0])
            # print(key,val,df[df['jid']==key],atoms)
            # energy,forces=get_alignn_forces(atoms)
            energy, forces, stress = get_pred(atoms=atoms)
            # print(key, val, energy, atoms.num_atoms)
            line = key + "," + str(entry.total_energy.values[0]*atoms.num_atoms)+","+str(energy) + "\n"
            print(line)
            f_e.write(line)
            line = (
                key
                + ","
                + str(";".join(map(str, np.array(entry.forces.values[0]).flatten())))
                +","
                + str(";".join(map(str, np.array(forces).flatten())))
                + "\n"
            )
            f_f.write(line)
            line = (
                key
                + ","
                + str(";".join(map(str, np.array(voigt_6_to_full_3x3_stress(entry.stresses.values[0])).flatten())))
                +","
                + str(";".join(map(str, np.array(voigt_6_to_full_3x3_stress(stress)).flatten())))
                + "\n"
            )
            f_s.write(line)
        f_e.close()
        f_f.close()
        f_s.close()
        zname = fname_e + ".zip"
        with zipfile.ZipFile(zname, "w") as myzip:
            myzip.write(fname_e)

        zname = fname_f + ".zip"
        with zipfile.ZipFile(zname, "w") as myzip:
            myzip.write(fname_f)

        zname = fname_s + ".zip"
        with zipfile.ZipFile(zname, "w") as myzip:
            myzip.write(fname_s)




In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error
%matplotlib inline
df = pd.read_csv('AI-MLFF-energy-mlearn_Si-test-mae.csv')
plt.plot(df.target,df.prediction,'.')
print(mean_absolute_error(df.target,df.prediction))

In [ ]:
df = pd.read_csv('AI-MLFF-forces-mlearn_Si-test-multimae.csv.zip')
target = np.concatenate(df.target.apply(lambda x:np.array(x.split(';'),dtype='float')).values)
pred= np.concatenate(df.prediction.apply(lambda x:np.array(x.split(';'),dtype='float')).values)
print(mean_absolute_error(target,pred))
plt.plot(target,pred,'.')


In [ ]:
# from ase.stress import voigt_6_to_full_3x3_stress

# df = pd.read_csv('AI-MLFF-stresses-mlearn_Si-test-multimae.csv.zip')
# target = np.concatenate(df.target.apply(lambda x:(np.array(x.split(';'),dtype='float')))).flatten()
# pred = 160*3*np.concatenate(df.prediction.apply(lambda x:(np.array(x.split(';'),dtype='float')))).flatten()
# print(mean_absolute_error(target,pred))
# plt.plot(target,pred,'.')


In [ ]:
from ase.eos import EquationOfState
from ase.units import kJ
from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms
import matplotlib.pyplot as plt
%matplotlib inline
atoms=Atoms.from_dict(get_jid_data(jid='JVASP-1002',dataset='dft_3d')['atoms']).make_supercell([2,2,2])# get_conventional_atoms
#dx=np.arange(-0.1, 0.1, 0.01)
dx=np.arange(-0.02, 0.02, 0.005)
y = []
vol = []
for i in dx:
    s1 = atoms.strain_atoms(i)
    energy, forces, stress = get_pred(atoms=s1)
    y.append(energy)
    vol.append(s1.volume)
x = np.array(dx)
y = np.array(y)
eos = EquationOfState(vol, y, eos="murnaghan")
v0, e0, B = eos.fit()
kv = B / kJ * 1.0e24  # , 'GPa')
print('kv',kv)
plt.plot(vol,y)
plt.xlabel('Volume ($\AA^3$)')
plt.ylabel('Total energy (eV)')


In [ ]:
from jarvis.io.vasp.inputs import Poscar
from ase.filters import ExpCellFilter
from ase.optimize.fire import FIRE
from jarvis.core.atoms import ase_to_atoms
from ase.filters import FrechetCellFilter

poscar="""System
1.0
5.0 0.0 0.0
-0.0 5.0 0.0
0.0 0.0 5.0
Si
8
direct
0.25 0.75 0.25 Si
0.0 0.0 0.5 Si
0.25 0.25 0.75 Si
0.0 0.5 0.0 Si
0.75 0.75 0.75 Si
0.5 0.0 0.0 Si
0.75 0.25 0.25 Si
0.5 0.5 0.5 Si
"""
# 5.49363
atoms=Poscar.from_string(poscar).atoms


def general_relaxer(atoms="", calculator="", fmax=0.05, steps=150,relax=True):
    ase_atoms = atoms.ase_converter()
    ase_atoms.calc = calculator
    if not relax:
         return ase_atoms.get_potential_energy()
    ase_atoms = ExpCellFilter(ase_atoms)
    #ase_atoms = FrechetCellFilter(ase_atoms)
    dyn = FIRE(ase_atoms)
    dyn.run(fmax=fmax, steps=steps)
    return ase_to_atoms(ase_atoms.atoms)
opt = general_relaxer(atoms=atoms,calculator=calc)
print(opt)


In [ ]:
#Check translationally equivariant
from ase import Atom, Atoms
import numpy as np
import matplotlib.pyplot as plt


A = 2.7223*np.array([[1,1,0],[1,0,1],[0,1,1]])
atoms_si = Atoms([Atom('Si', (0, 0, 0)), Atom('Si', (0.25, 0.25, 0.25))], cell = A, pbc=True)

atoms_si.set_tags(np.ones(len(atoms_si)))
atoms_si.calc = calc

atoms_si.set_scaled_positions(np.array([[0,0,0],[0.25, 0.25, 0.25]]))

#"""
en = []
dist = []
for a in np.linspace(-0.1,  0.1, 51):
    atoms_si.set_scaled_positions(np.array([[a,0,0],[0.25, 0.25, 0.25]]))
    e = atoms_si.get_potential_energy()
    en.append(e)
    dist.append(np.sign(a)*np.sqrt(np.sum(np.dot(A, [a,0,0])**2)))
en_111_more = []
dist_111_more = []
for a in np.linspace(-0.1,  0.1, 201):
    atoms_si.set_scaled_positions(np.array([[a,a,a],[0.25, 0.25, 0.25]]))
    e = atoms_si.get_potential_energy()
    en_111_more.append(e)
    dist_111_more.append(np.sign(a)*np.sqrt(np.sum(np.dot(A, [a,a,a])**2)))
plt.plot(dist,en)
plt.plot(dist_111_more,en_111_more)
#plt.ylim([-10.7,-10])
# plt.savefig('inv.png')
# plt.close()
#"""




Scaling

In [ ]:
"""
ALIGNN-FF scaling benchmark — pure_torch (no DGL required)
Trained model: OutputDir/ (alignn_atomwise_pure)

Runs on any platform with PyTorch + CUDA, including Colab free tier.
No DGL dependency: builder returns TorchGraph, model consumes TorchGraph.
"""
import os
os.environ.setdefault("CUDA_DEVICE_ORDER", "PCI_BUS_ID")

import time
import gc
import numpy as np
import matplotlib.pyplot as plt
import torch

from ase import Atoms as ASEAtoms, Atom
from ase.build.supercells import make_supercell

from jarvis.core.atoms import ase_to_atoms
from jarvis.db.jsonutils import loadjson

from alignn.torch_graph_builder import build_pure_torch_graph
from alignn.models.alignn_atomwise_pure import (
    ALIGNNAtomWisePure, ALIGNNAtomWisePureConfig,
)

# ---------- Config ----------
MODEL_PATH = "OutputDir"
SIZES      = np.arange(1, 20)      # N = 2 * i^3
N_REPEATS  = 3
DEVICE     = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ---------- Load model ----------
config = loadjson(os.path.join(MODEL_PATH, "config.json"))
config["model"]["calculate_gradient"] = False
config["model"]["stresswise_weight"]  = 0
config["model"]["graphwise_weight"]   = 1.0
config["model"]["atomwise_weight"]    = 0
config["model"]["gradwise_weight"]    = 0

CUTOFF        = config["cutoff"]
MAX_NEIGHBORS = config["max_neighbors"]
ATOM_FEATURES = config["atom_features"]
R3            = config.get("three_body_cutoff", 3.5)
print(f"device={DEVICE}")
print(f"cutoff={CUTOFF}  max_neighbors={MAX_NEIGHBORS}  "
      f"atom_features={ATOM_FEATURES}  three_body_cutoff={R3}")

model = ALIGNNAtomWisePure(ALIGNNAtomWisePureConfig(**config["model"]))
model.load_state_dict(
    torch.load(os.path.join(MODEL_PATH, "best_model.pt"),
               map_location=DEVICE, weights_only=False)
)
model.to(DEVICE).eval()


def build_graph(jarvis_atoms):
    """Build (g, lg) as TorchGraph objects directly on the target device.

    No DGL involved. The model's forward() accepts TorchGraph natively.
    """
    return build_pure_torch_graph(
        atoms=jarvis_atoms,
        two_body_cutoff=CUTOFF,
        three_body_cutoff=R3,
        max_neighbors=MAX_NEIGHBORS,
        atom_features=ATOM_FEATURES,
        device=DEVICE,                  # build on GPU
        use_matscipy_topology=False,    # pure-torch fallback (GPU-native)
        compute_line_graph=True,
    )


def run_once(ase_atoms):
    """Return (t_graph, t_infer, energy_per_atom)."""
    j_atoms = ase_to_atoms(ase_atoms)

    # --- graph construction (GPU work timed with sync) ---
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t0 = time.time()
    g, lg = build_graph(j_atoms)
    cell = torch.tensor(np.array(j_atoms.lattice_mat),
                        dtype=torch.float32, device=DEVICE)
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t_graph = time.time() - t0

    # --- inference ---
    t1 = time.time()
    with torch.no_grad():
        out = model((g, lg, cell))
        energy = out["out"].detach().cpu().item()
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    t_infer = time.time() - t1

    del g, lg, cell, out
    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

    return t_graph, t_infer, energy


# ---------- Primitive Si cell (2 atoms) ----------
A = 2.7223 * np.array([[1, 1, 0], [1, 0, 1], [0, 1, 1]])
prim = ASEAtoms(
    [Atom("Si", (0, 0, 0)), Atom("Si", (0.25, 0.25, 0.25))],
    cell=A, pbc=True,
)

# ---------- Warmup (first GPU call compiles kernels, autotunes) ----------
print("Warming up...")
for _ in range(2):
    _ = run_once(make_supercell(prim, np.eye(3, dtype=int) * 2))

# ---------- Timing loop ----------
natoms = []
t_graph_med, t_graph_lo, t_graph_hi = [], [], []
t_infer_med, t_infer_lo, t_infer_hi = [], [], []

for i in SIZES:
    sc = make_supercell(prim, np.eye(3, dtype=int) * i)
    N  = len(sc)

    gs, is_ = [], []
    try:
        for _ in range(N_REPEATS):
            tg, ti, e = run_once(sc)
            gs.append(tg); is_.append(ti)
    except torch.cuda.OutOfMemoryError as err:
        print(f"\n[OOM at i={i}, N={N}] {err}")
        print("Stopping scan. Reduce SIZES or lower chunk_size "
              "in torch_neighbor_list.\n")
        if DEVICE.type == "cuda":
            torch.cuda.empty_cache()
        break

    natoms.append(N)
    t_graph_med.append(np.median(gs)); t_graph_lo.append(min(gs)); t_graph_hi.append(max(gs))
    t_infer_med.append(np.median(is_)); t_infer_lo.append(min(is_)); t_infer_hi.append(max(is_))

    mem_gb = (torch.cuda.max_memory_allocated() / 1e9
              if DEVICE.type == "cuda" else 0.0)
    print(f"i={i:2d} | N={N:6d} | E/atom={e:+.4f} eV | "
          f"graph={np.median(gs):.4f}s  infer={np.median(is_):.4f}s  "
          f"total={np.median(gs)+np.median(is_):.4f}s  "
          f"peak_mem={mem_gb:.2f} GB")

    if DEVICE.type == "cuda":
        torch.cuda.reset_peak_memory_stats()

    np.savez("alignn_scaling_gpu.npz",
             natoms=np.array(natoms),
             t_graph=np.array(t_graph_med), t_graph_lo=np.array(t_graph_lo), t_graph_hi=np.array(t_graph_hi),
             t_infer=np.array(t_infer_med), t_infer_lo=np.array(t_infer_lo), t_infer_hi=np.array(t_infer_hi))

if len(natoms) < 3:
    raise SystemExit("Not enough data points for analysis. Check for early OOM.")

# ---------- Analysis ----------
natoms   = np.array(natoms)
t_graph  = np.array(t_graph_med)
t_infer  = np.array(t_infer_med)
t_total  = t_graph + t_infer

def fit_power(N, t, n_min=50):
    m = N >= n_min
    if m.sum() < 3:
        return None, None
    a, c = np.polyfit(np.log(N[m]), np.log(t[m]), 1)
    return a, np.exp(c)

a_g, C_g = fit_power(natoms, t_graph)
a_i, C_i = fit_power(natoms, t_infer)
a_t, C_t = fit_power(natoms, t_total)

print(f"\nScaling exponents (fit for N>=50):")
if a_g is not None: print(f"  graph:     t ~ {C_g:.3e} * N^{a_g:.3f}")
if a_i is not None: print(f"  inference: t ~ {C_i:.3e} * N^{a_i:.3f}")
if a_t is not None: print(f"  total:     t ~ {C_t:.3e} * N^{a_t:.3f}")
print(f"Peak inference throughput: {(natoms/t_infer).max():.0f} atoms/s "
      f"at N={natoms[np.argmax(natoms/t_infer)]}")
print(f"Peak total throughput:     {(natoms/t_total).max():.0f} atoms/s "
      f"at N={natoms[np.argmax(natoms/t_total)]}")
print(f"Largest system completed:  N={natoms[-1]}, t_total={t_total[-1]:.3f}s")

# ---------- Plot ----------
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
n_fit = np.logspace(np.log10(natoms.min()), np.log10(natoms.max()), 100)

# (1) Component scaling
ax = axes[0]
ax.errorbar(natoms, t_graph,
            yerr=[t_graph - np.array(t_graph_lo), np.array(t_graph_hi) - t_graph],
            fmt="s", color="#ff7f0e", markersize=6, capsize=3,
            label="Graph construction (GPU)")
ax.errorbar(natoms, t_infer,
            yerr=[t_infer - np.array(t_infer_lo), np.array(t_infer_hi) - t_infer],
            fmt="o", color="#1f77b4", markersize=6, capsize=3,
            label="Model inference (GPU)")
ax.plot(natoms, t_total, "-^", color="#2ca02c", markersize=6, label="Total")
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Number of atoms"); ax.set_ylabel("Time (s)")
ax.set_title("Component scaling — pure_torch on GPU (no DGL)")
ax.grid(True, which="both", alpha=0.3)
ax.legend(frameon=False)

# (2) Total with fit + reference slopes
ax = axes[1]
ax.loglog(natoms, t_total, "o", color="#2ca02c", markersize=8,
          label="Total (measured)")
if a_t is not None:
    ax.loglog(n_fit, C_t * n_fit**a_t, "--", color="#d62728",
              label=f"Fit: $N^{{{a_t:.2f}}}$")
ax.loglog(n_fit, t_total[0] * (n_fit/natoms[0])**1.0, ":", color="gray",
          alpha=0.7, label="$N^1$")
ax.loglog(n_fit, t_total[0] * (n_fit/natoms[0])**2.0, ":", color="black",
          alpha=0.5, label="$N^2$")
ax.set_xlabel("Number of atoms"); ax.set_ylabel("Total time (s)")
title_slope = f"{a_t:.2f}" if a_t is not None else "n/a"
ax.set_title(f"Total scaling  (slope = {title_slope})")
ax.grid(True, which="both", alpha=0.3); ax.legend(frameon=False)

# (3) Throughput
ax = axes[2]
ax.semilogx(natoms, natoms/t_infer, "o-", color="#1f77b4", label="Inference only")
ax.semilogx(natoms, natoms/t_total, "s-", color="#2ca02c", label="Total (incl. graph)")
ax.set_xlabel("Number of atoms"); ax.set_ylabel("Throughput (atoms / s)")
ax.set_title("Throughput vs system size")
ax.grid(True, which="both", alpha=0.3); ax.legend(frameon=False)

plt.tight_layout()
plt.savefig("alignn_scaling_gpu.png", dpi=200)
plt.show()

Melt quench

In [ ]:
%%time
import ase,os
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.nvtberendsen import NVTBerendsen
from jarvis.db.figshare import get_jid_data
from jarvis.core.atoms import Atoms, ase_to_atoms
calc = AlignnAtomwiseCalculator(path='OutputDir',device=device, force_mult_batchsize=False)
!rm */*.traj
def ensure_cell_size(ase_atoms, min_size):
    """Ensure that all cell dimensions are at least min_size."""
    cell_lengths = ase_atoms.get_cell().lengths()
    scale_factors = np.ceil(min_size / cell_lengths).astype(int)
    supercell_dims = [max(1, scale) for scale in scale_factors]
    return supercell_dims

calculator = calc
jid = 'JVASP-1002'
atoms=Atoms.from_dict(get_jid_data(jid=jid,dataset='dft_3d')['atoms']).get_conventional_atoms
relaxed_atoms=atoms
ase_atoms = atoms.ase_converter()
dim = ensure_cell_size(ase_atoms, min_size=10.0)
supercell = relaxed_atoms.make_supercell_matrix(dim)
ase_atoms = supercell.ase_converter()
ase_atoms.calc = calculator

dt = 1 * ase.units.fs
temp0, nsteps0 = 3500, 1000
temp1, nsteps1 = 300, 2000
taut = 20 * ase.units.fs
if not os.path.exists("output_melt"):
    os.mkdir("output_melt")
trj = os.path.join("output_melt", f"{jid}_melt.traj")

# Initialize velocities and run the first part of the MD simulation


MaxwellBoltzmannDistribution(ase_atoms, temp0 * ase.units.kB)
dyn = NVTBerendsen(ase_atoms, dt, temp0, taut=taut, trajectory=trj)

def myprint():
    message = f"time={dyn.get_time() / ase.units.fs: 5.0f} fs T={ase_atoms.get_temperature(): 3.0f} K"
    print(message)
dyn.attach(myprint, interval=20)
dyn.run(nsteps0)

# Cool down to room temperature
dyn.set_temperature(temp1)
dyn.run(nsteps1)

# Convert back to JARVIS atoms and save the final structure
final_atoms = ase_to_atoms(ase_atoms)
poscar_filename = os.path.join(
    "output_melt", f"POSCAR_{jid}_quenched_{'alignn'}.vasp"
)
from ase.io import write

write(poscar_filename, final_atoms.ase_converter(), format="vasp")
# self.log(f"MD simulation completed. Final structure saved to {poscar_filename}")
# self.job_info["quenched_atoms"] = final_atoms.to_dict()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from jarvis.core.atoms import Atoms
from ase.io.trajectory import Trajectory
from jarvis.core.atoms import ase_to_atoms
from ase.visualize.plot import plot_atoms
from matplotlib.animation import FuncAnimation
from ase.geometry.analysis import Analysis


traj = Trajectory('output_melt/JVASP-1002_melt.traj')


fig, ax = plt.subplots(1, 3, figsize=(9,3), tight_layout=True)

t = np.arange(nsteps0+nsteps1+1) * dt
temp = [atoms.get_temperature() for atoms in traj]

nframes = 20



def update(iframe):
    idx = int((nsteps0+nsteps1)*iframe/nframes)

    ax[0].clear()
    ax[0].set_title('Temperature')
    ax[0].set_xlabel('time (fs)')
    ax[0].set_ylabel('T (K)')
    ax[0].plot(t, temp)
    ax[0].plot(t[idx], temp[idx], marker='X', markersize=10)

    ax[1].clear()
    ax[1].set_title('Structure')
    ax[1].axis('off')
    plot_atoms(traj[idx], ax=ax[1], rotation='45x,45y')

    distribution, distance = Analysis(traj[idx]).get_rdf(rmax=5., nbins=100, return_dists=True)[0]
    ax[2].clear()
    ax[2].set_title('RDF')
    ax[2].set_ylim((0,10))
    ax[2].set_xlabel('distance (A))')
    ax[2].set_ylabel('distribution')
    ax[2].plot(distance, distribution, color='darkblue')
ani = FuncAnimation(fig, update, np.arange(nframes), blit=False, interval=250.)
ani.save('ani_si_2k.gif', writer="imagemagick")
plt.close()

atoms=ase_to_atoms(traj[-1])
print(atoms)
dists=[]
for i in atoms.get_all_neighbors(r=10):
    for j in i:
      dists.append(j[2])
dists=np.array(dists)
hist, bins = np.histogram(dists, bins=np.arange(0.1, 10.2, 0.1))
number_density = atoms.num_atoms / atoms.volume
shell_vol = (
            4.0
            / 3.0
            * np.pi
            * (np.power(bins[1:], 3) - np.power(bins[:-1], 3))
        )
rdf = (
            hist / shell_vol / number_density / atoms.num_atoms
        )
plt.plot(bins[:-1], rdf)
plt.xlim([0,10])
plt.xlabel('Distance (Angstrom)')
plt.ylabel('RDF')
# plt.savefig('rdf.png')
# plt.close()
plt.show()


In [ ]:
from IPython.display import Image, display

display(Image(filename='ani_si_2k.gif'))


In [ ]:
from ase.visualize import view
view(atoms.ase_converter(),viewer='x3d')

In [ ]:
atoms.plot_atoms()


Training analysis

In [ ]:
from jarvis.db.jsonutils import loadjson
from matplotlib.gridspec import GridSpec
import numpy as np
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
import os
%matplotlib inline
# Plot training hostory for validation set
dir_name = "OutputDir"
json_path = os.path.join(dir_name, "history_val.json")
v = loadjson(json_path)
ens = []
fs = []
for i in v:
    ens.append(i[0])
    fs.append(i[1])
the_grid = GridSpec(1, 2)
plt.rcParams.update({"font.size": 18})
plt.figure(figsize=(12, 5))
plt.subplot(the_grid[0])
plt.title("(a) Energy")
plt.plot(ens)
plt.xlabel("Epochs")
plt.ylabel("eV")
plt.subplot(the_grid[1])
plt.title("(b) Forces")
plt.plot(fs)
plt.xlabel("Epochs")
plt.ylabel("eV/A")
plt.tight_layout()
plt.show()
# plt.savefig("history.png")
# plt.close()
print(v[-1])

In [ ]:
the_grid = GridSpec(1, 2)
json_path = os.path.join(dir_name, "Val_results.json")
test = loadjson(json_path)
plt.rcParams.update({"font.size": 18})
plt.figure(figsize=(12, 5))
plt.subplot(the_grid[0])
xx = []
yy = []
factor = 1
for i in test:
    for j, k in zip(i["target_out"], i["pred_out"]):
        xx.append(j)
        yy.append(k)
xx = np.array(xx) * factor
yy = np.array(yy) * factor

x_bar = np.mean(xx)
baseline_mae = mean_absolute_error(
    np.array(xx),
    np.array([x_bar for i in range(len(xx))]),
)
print("Val")
print("Baseline MAE: eV", baseline_mae)
print("MAE eV", mean_absolute_error(xx, yy))

plt.plot(xx, yy, ".")
plt.ylabel("ALIGNN Energy (eV)")
plt.xlabel("DFT Energy (eV)")
plt.subplot(the_grid[1])
xx = []
yy = []
for i in test:
    for j, k in zip(i["target_grad"], i["pred_grad"]):
        for m, n in zip(j, k):
            xx.append(m)
            yy.append(n)
xx = np.array(xx) * factor
yy = np.array(yy) * factor

x_bar = np.mean(xx)
baseline_mae = mean_absolute_error(
    np.array(xx),
    np.array([x_bar for i in range(len(xx))]),
)
print("Test")
print("Baseline MAE: eV/A", baseline_mae)
print("MAE eV/A", mean_absolute_error(xx, yy))
plt.scatter(xx, yy, c="blueviolet", s=10, alpha=0.5)

plt.scatter(xx, yy, c="blueviolet", s=10, alpha=0.5)
plt.ylabel("ALIGNN Force (eV/A)")
plt.xlabel("DFT Force (eV/A)")
plt.tight_layout()
plt.show()
# plt.savefig("val.png")
# plt.close()


Equation of state

## Phonon bandstructure

In [ ]:
# import locale
# locale.getpreferredencoding = lambda: "UTF-8"

In [ ]:
!pip install phonopy

In [ ]:
  import argparse
  import numpy as np
  import plotly.graph_objects as go
  from ase import Atoms as AseAtoms
  from ase.filters import ExpCellFilter
  from ase.optimize import FIRE
  from phonopy import Phonopy
  from phonopy.structure.atoms import PhonopyAtoms

  from jarvis.core.atoms import Atoms, ase_to_atoms
  from jarvis.core.kpoints import Kpoints3D as Kpoints
  from jarvis.db.figshare import get_jid_data

  from alignn.ff.ff import AlignnAtomwiseCalculator


  def relax(atoms: Atoms, calc, fmax=0.05, steps=100):
      a = atoms.ase_converter()
      a.calc = calc
      FIRE(ExpCellFilter(a), logfile=None).run(fmax=fmax, steps=steps)
      return ase_to_atoms(a)


  def compute_phonons(atoms, calc, dim=(2, 2, 2), distance=0.03, line_density=20):
      bulk = PhonopyAtoms(
          symbols=atoms.elements,
          scaled_positions=atoms.frac_coords,
          cell=atoms.lattice_mat,
      )
      ph = Phonopy(bulk, [[dim[0], 0, 0], [0, dim[1], 0], [0, 0, dim[2]]])
      ph.generate_displacements(distance=distance)

      forces_set = []
      # `.supercells_with_displacements` is the modern property;
      # the getter method was removed in newer phonopy versions.
      supercells = ph.supercells_with_displacements
      for sc in supercells:
          a = AseAtoms(
              symbols=list(sc.symbols),
              scaled_positions=np.array(sc.scaled_positions),
              cell=np.array(sc.cell),
              pbc=True,
          )
          a.calc = calc
          F = np.array(a.get_forces())
          F -= F.mean(axis=0)
          forces_set.append(F)
      ph.produce_force_constants(forces=forces_set)

      kp = Kpoints().kpath(atoms, line_density=line_density)
      q_points = np.array(kp.kpts)
      labels_raw = kp.labels

      freqs, distances, tick_pos, tick_labels = [], [], [], []
      last_kstr = None
      d = 0.0
      for i, (q, lbl) in enumerate(zip(q_points, labels_raw)):
          if i > 0:
              d += np.linalg.norm(q_points[i] - q_points[i - 1])
          distances.append(d)
          freqs.append(ph.get_frequencies(q))
          kstr = ",".join(f"{x:.6f}" for x in q)
          if lbl and kstr != last_kstr:
              tick_pos.append(d)
              tick_labels.append(lbl)
              last_kstr = kstr
      #return np.array(distances), np.array(freqs), tick_pos, tick_labels
      return np.array(distances), np.array(freqs), tick_pos, tick_labels, ph


  def plot(distances, freqs, tick_pos, tick_labels, title=""):
      fig = go.Figure()
      for b in range(freqs.shape[1]):
          fig.add_trace(go.Scatter(
              x=distances, y=freqs[:, b], mode="lines",
              line=dict(color="steelblue", width=1.5),
              showlegend=False, hovertemplate="d=%{x:.3f}<br>ν=%{y:.3f} THz",
          ))
      fig.add_hline(y=0, line=dict(color="black", dash="dot", width=1))
      for xp in tick_pos[1:-1]:
          fig.add_vline(x=xp, line=dict(color="lightgray", width=1))
      fig.update_layout(
          title=title, width=800, height=500,
          xaxis=dict(
              title="Wave vector",
              tickmode="array",
              tickvals=tick_pos,
              ticktext=[l.replace("\\Gamma", "Γ") for l in tick_labels],
              showgrid=False, zeroline=False,
          ),
          yaxis=dict(title="Frequency (THz)", gridcolor="lightgray"),
          template="plotly_white",
      )
      return fig


  # ---- run ----
  calc = AlignnAtomwiseCalculator(path="OutputDir", force_mult_batchsize=False)
  jid = "JVASP-1002"
  dim = [2, 2, 2]
  out = "phonons.html"

  d = get_jid_data(jid=jid, dataset="dft_3d")
  atoms = Atoms.from_dict(d["atoms"]).get_conventional_atoms
  print(f"loaded {jid}  {atoms.composition.reduced_formula}  N={atoms.num_atoms}")

  relaxed = relax(atoms, calc)
  prim = relaxed.get_primitive_atoms
  print(f"relaxed → primitive N={prim.num_atoms}")

  distances, freqs, tick_pos, tick_labels,ph = compute_phonons(prim, calc, dim=tuple(dim))
  n_imag = int((freqs < -0.05).sum())
  print(f"bands computed: {freqs.shape[1]}  imag modes (<-0.05 THz): {n_imag}")

  fig = plot(distances, freqs, tick_pos, tick_labels,
             title=f"Phonon bands — {atoms.composition.reduced_formula} ({jid})  supercell {dim}")
  fig.write_html(out)
  fig.show()   # add this in a notebook to display inline
  print(f"wrote {out}")

In [ ]:
  # --- Harmonic thermal properties (Cv, S, F, ZPE) on a q-mesh ---
  import numpy as np
  import plotly.graph_objects as go
  from plotly.subplots import make_subplots

  ph.run_mesh([20, 20, 20])                       # dense BZ sampling
  ph.run_thermal_properties(t_min=0, t_max=1000, t_step=10)

  td = ph.get_thermal_properties_dict()
  T   = td["temperatures"]            # K
  F   = td["free_energy"]             # kJ/mol  (Helmholtz, per unit cell)
  S   = td["entropy"]                 # J/K/mol
  Cv  = td["heat_capacity"]           # J/K/mol

  # Per-atom normalization (phonopy returns per-unit-cell quantities)
  N = len(ph.primitive)
  Cv_at = Cv / N
  S_at  = S  / N
  F_at  = F  / N

  # Zero-point energy from 0 K free energy (in kJ/mol per primitive cell)
  ZPE = F[0]
  print(f"ZPE per primitive cell: {ZPE:.4f} kJ/mol  ({ZPE*1000/96.485:.4f} eV)")
  print(f"Cv @ 300 K: {Cv_at[np.argmin(abs(T-300))]:.3f} J/K/mol-atom  "
        f"(Dulong-Petit limit: 24.94)")

  # --- Plot ---
  fig = make_subplots(rows=1, cols=3, subplot_titles=("Cv", "Entropy S", "Free energy F"))
  fig.add_trace(go.Scatter(x=T, y=Cv_at, name="Cv"), row=1, col=1)
  fig.add_hline(y=24.94, line=dict(dash="dot", color="gray"), row=1, col=1)   # 3R limit
  fig.add_trace(go.Scatter(x=T, y=S_at,  name="S"),  row=1, col=2)
  fig.add_trace(go.Scatter(x=T, y=F_at,  name="F"),  row=1, col=3)
  fig.update_xaxes(title_text="Temperature (K)")
  fig.update_yaxes(title_text="Cv (J/K/mol-atom)", row=1, col=1)
  fig.update_yaxes(title_text="S (J/K/mol-atom)",  row=1, col=2)
  fig.update_yaxes(title_text="F (kJ/mol-atom)",   row=1, col=3)
  fig.update_layout(template="plotly_white", width=1100, height=400, showlegend=False)
  fig.write_html("thermal.html")
  fig.show()

In [ ]:
!pip install phono3py

You can compare the bandstructure here: https://www.ctcms.nist.gov/~knc6/static/JARVIS-DFT/JVASP-1002.xml#elastic_tensor

In [ ]:
"""Lattice thermal conductivity κ(T) and 3-phonon properties from ALIGNN-FF.

Uses phono3py for anharmonic (3rd-order) force constants, then solves the
linearized Boltzmann transport equation under the relaxation-time
approximation (RTA).

WARNING: this is expensive. For Si primitive + 2x2x2 supercell (16 atoms),
phono3py generates ~111 displacements for fc3 + ~1 for fc2 ~~ hundreds
of force evaluations. Allow 10-30 min on a single GPU.

Usage:
    python kappa_plotly.py --model-dir OutputDir --jid JVASP-1002 \\
        --dim 2 2 2 --mesh 11 11 11
"""
from __future__ import annotations
import argparse
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from ase import Atoms as AseAtoms
from ase.filters import ExpCellFilter
from ase.optimize import FIRE
from phono3py import Phono3py
from phonopy.structure.atoms import PhonopyAtoms

from jarvis.core.atoms import Atoms, ase_to_atoms
from jarvis.db.figshare import get_jid_data

from alignn.ff.ff import AlignnAtomwiseCalculator


def relax(atoms, calc, fmax=0.05, steps=100):
    a = atoms.ase_converter()
    a.calc = calc
    FIRE(ExpCellFilter(a), logfile=None).run(fmax=fmax, steps=steps)
    return ase_to_atoms(a)


def forces_on_supercell(sc, calc):
    a = AseAtoms(
        symbols=list(sc.symbols),
        scaled_positions=np.array(sc.scaled_positions),
        cell=np.array(sc.cell),
        pbc=True,
    )
    a.calc = calc
    F = np.array(a.get_forces())
    F -= F.mean(axis=0)
    return F


def run_phono3py(atoms, calc, dim=(2, 2, 2), dim_fc2=None, distance=0.03):
    """Build Phono3py, compute fc3 (and fc2) via ALIGNN-FF."""
    bulk = PhonopyAtoms(
        symbols=atoms.elements,
        scaled_positions=atoms.frac_coords,
        cell=atoms.lattice_mat,
    )
    # fc2 supercell can be larger than fc3 supercell for a better harmonic part
    if dim_fc2 is None:
        dim_fc2 = dim
    ph3 = Phono3py(
        bulk,
        supercell_matrix=[[dim[0], 0, 0], [0, dim[1], 0], [0, 0, dim[2]]],
        phonon_supercell_matrix=[[dim_fc2[0], 0, 0], [0, dim_fc2[1], 0], [0, 0, dim_fc2[2]]],
    )
    ph3.generate_displacements(distance=distance)

    # fc3 supercells
    sc3 = ph3.supercells_with_displacements
    n3 = len(sc3)
    print(f"fc3 displacements: {n3}")
    fset3 = []
    for i, sc in enumerate(sc3):
        if sc is None:                                  # symmetry-reduced skip
            fset3.append(None); continue
        F = forces_on_supercell(sc, calc)
        fset3.append(F)
        if (i + 1) % 10 == 0 or i == n3 - 1:
            print(f"  fc3 {i+1}/{n3}")
    ph3.forces = fset3

    # fc2 supercells (phonon supercells)
    sc2 = ph3.phonon_supercells_with_displacements
    n2 = len(sc2)
    print(f"fc2 displacements: {n2}")
    fset2 = []
    for i, sc in enumerate(sc2):
        if sc is None:
            fset2.append(None); continue
        F = forces_on_supercell(sc, calc)
        fset2.append(F)
    ph3.phonon_forces = fset2

    ph3.produce_fc2(symmetrize_fc2=True)
    ph3.produce_fc3(symmetrize_fc3r=True)
    return ph3


def compute_kappa(ph3, mesh=(11, 11, 11), t_min=100, t_max=1000, t_step=50,
                  boundary_mfp=None):
    """Run RTA thermal conductivity. Returns (T, kappa_xx_yy_zz_xy_xz_yz)."""
    ph3.mesh_numbers = list(mesh)
    ph3.init_phph_interaction()
    temperatures = np.arange(t_min, t_max + 1, t_step)
    ph3.run_thermal_conductivity(
        temperatures=temperatures,
        is_isotope=False,
        boundary_mfp=boundary_mfp,      # µm; None = infinite sample
        write_kappa=False,
    )
    tc = ph3.thermal_conductivity
    T = np.asarray(tc.temperatures)
    kappa = np.asarray(tc.kappa)   # shape (1, nT, 6): xx yy zz yz xz xy
    return T, kappa[0]


def plot_kappa(T, kappa, thermo=None, title=""):
    # trace of kappa = average of diagonal components
    kappa_avg = kappa[:, :3].mean(axis=1)

    fig = make_subplots(
        rows=1, cols=2,
        subplot_titles=("Thermal conductivity κ(T)",
                        "Log-log view"),
        horizontal_spacing=0.12,
    )
    for i, lbl in enumerate(["κ_xx", "κ_yy", "κ_zz"]):
        fig.add_trace(go.Scatter(x=T, y=kappa[:, i], mode="lines+markers",
                                 name=lbl, line=dict(width=1.5)),
                      row=1, col=1)
    fig.add_trace(go.Scatter(x=T, y=kappa_avg, mode="lines",
                             name="κ_avg", line=dict(color="black", dash="dash")),
                  row=1, col=1)

    fig.add_trace(go.Scatter(x=T, y=kappa_avg, mode="lines+markers",
                             name="κ_avg (log-log)",
                             line=dict(color="steelblue", width=1.5),
                             showlegend=False),
                  row=1, col=2)
    fig.update_xaxes(type="log", title_text="T (K)", row=1, col=2)
    fig.update_yaxes(type="log", title_text="κ (W/m·K)", row=1, col=2)

    fig.update_xaxes(title_text="T (K)", row=1, col=1)
    fig.update_yaxes(title_text="κ (W/m·K)", row=1, col=1)
    fig.update_layout(template="plotly_white", title=title,
                     width=1100, height=450)
    return fig


def print_summary(T, kappa, name=""):
    kappa_avg = kappa[:, :3].mean(axis=1)
    i300 = int(np.argmin(abs(T - 300)))
    print(f"\n── κ summary {name} ──")
    print(f"κ_xx, κ_yy, κ_zz (300 K) = {kappa[i300,0]:.2f}, "
          f"{kappa[i300,1]:.2f}, {kappa[i300,2]:.2f} W/m·K")
    print(f"κ_avg(300 K) = {kappa_avg[i300]:.2f} W/m·K")
    # T-scaling: fit log(κ) = a - p*log(T) in high-T regime
    hi = T > 300
    if hi.sum() > 2:
        slope = np.polyfit(np.log(T[hi]), np.log(kappa_avg[hi]), 1)[0]
        print(f"high-T exponent p (κ ~ T^-p): p ≈ {-slope:.2f}  "
              f"(3-phonon expects ~1)")



# ap = argparse.ArgumentParser()
# ap.add_argument("--model-dir", default="OutputDir")
# ap.add_argument("--jid", default="JVASP-1002")
# ap.add_argument("--dim", type=int, nargs=3, default=[2, 2, 2],
#                 help="fc3 supercell")
# ap.add_argument("--dim-fc2", type=int, nargs=3, default=None,
#                 help="fc2 supercell (default = fc3 supercell)")
# ap.add_argument("--mesh", type=int, nargs=3, default=[11, 11, 11])
# ap.add_argument("--t-max", type=float, default=1000.0)
# ap.add_argument("--out", default="kappa.html")
# args = ap.parse_args()

calc = AlignnAtomwiseCalculator(path="OutputDir", force_mult_batchsize=False)
jid = "JVASP-1002"
dim = [2,2,2]
out = "phonons.html"
mesh=[11, 11, 11]
t_max=1000.0
out = "kappa.html"
dim_fc2=None
model_dir=  "OutputDir"

# d = get_jid_data(jid=jid, dataset="dft_3d")
# atoms = Atoms.from_dict(d["atoms"]).get_conventional_atoms

calc = AlignnAtomwiseCalculator(path=model_dir, force_mult_batchsize=False)
d = get_jid_data(jid=jid, dataset="dft_3d")
atoms = Atoms.from_dict(d["atoms"]).get_conventional_atoms
formula = atoms.composition.reduced_formula
print(f"loaded {jid}  {formula}  N={atoms.num_atoms}")

relaxed = relax(atoms, calc)
prim = relaxed.get_primitive_atoms
print(f"relaxed → primitive N={prim.num_atoms}")

ph3 = run_phono3py(
    prim, calc,
    dim=tuple(dim),
    dim_fc2=tuple(dim_fc2) if dim_fc2 else None,
)
T, kappa = compute_kappa(ph3, mesh=tuple(mesh), t_max=t_max)
print_summary(T, kappa, formula)

fig = plot_kappa(T, kappa,
                title=f"{formula} ({jid}) — κ(T) via phono3py+ALIGNN-FF "
                      f"[fc3 {dim}, mesh {mesh}]")
fig.write_html(out)
fig.show()
print(f"\nwrote {out}")
# NOTE: Currently underestimated

Elastic constants

In [ ]:
!pip install -q elastic

In [ ]:
from elastic import get_elementary_deformations, get_elastic_tensor
import elastic
import ase
from jarvis.core.atoms import ase_to_atoms
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import get_jid_data

ph_path='OutputDir'
atoms=Atoms.from_dict(get_jid_data(jid='JVASP-1002',dataset='dft_3d')['atoms']).get_conventional_atoms
ase_atoms = atoms.ase_converter()
ase_atoms.calc = calc
systems = get_elementary_deformations(ase_atoms)
cij_order = elastic.elastic.get_cij_order(ase_atoms)
Cij, Bij = get_elastic_tensor(ase_atoms, systems)
for i, j in zip(cij_order, Cij):
    print(i, j / ase.units.GPa)

In [ ]:
#Check translationally equivariant
from ase import Atom, Atoms
import numpy as np
import matplotlib.pyplot as plt


A = 2.7223*np.array([[1,1,0],[1,0,1],[0,1,1]])
atoms_si = Atoms([Atom('Si', (0, 0, 0)), Atom('Si', (0.25, 0.25, 0.25))], cell = A, pbc=True)

atoms_si.set_tags(np.ones(len(atoms_si)))
atoms_si.calc = calc

atoms_si.set_scaled_positions(np.array([[0,0,0],[0.25, 0.25, 0.25]]))

#"""
en = []
dist = []
for a in np.linspace(-0.1,  0.1, 51):
    atoms_si.set_scaled_positions(np.array([[a,0,0],[0.25, 0.25, 0.25]]))
    e = atoms_si.get_potential_energy()
    en.append(e)
    dist.append(np.sign(a)*np.sqrt(np.sum(np.dot(A, [a,0,0])**2)))
en_111_more = []
dist_111_more = []
for a in np.linspace(-0.1,  0.1, 201):
    atoms_si.set_scaled_positions(np.array([[a,a,a],[0.25, 0.25, 0.25]]))
    e = atoms_si.get_potential_energy()
    en_111_more.append(e)
    dist_111_more.append(np.sign(a)*np.sqrt(np.sum(np.dot(A, [a,a,a])**2)))
plt.plot(dist,en)
plt.plot(dist_111_more,en_111_more)
#plt.ylim([-10.7,-10])
# plt.savefig('kevin.png')
# plt.close()
#"""




In [ ]:
#Check rotationally equivariant
atoms_si = Atoms([Atom('Si', (0, 0, 0)), Atom('Si', (0.25, 0.25, 0.25))], cell = A, pbc=True)
atoms_si.calc = calc
#translations
en_trans = []
for a in np.linspace(-0.1,  0.1, 51):
    atoms_si.set_scaled_positions(np.array([[a,a,a],[a+0.25, a+0.25, a+0.25]]))
    e = atoms_si.get_potential_energy()
    en_trans.append(e)
plt.plot(en_trans)

In [ ]:
en_trans

In [ ]:
"""NEMD thermal conductivity via Müller-Plathe reverse method.

Imposes a heat flux by periodically swapping kinetic energies between
hot and cold slabs, then measures the resulting temperature gradient.
κ = J_z / (dT/dz)

For Si at 300 K with ALIGNN-FF, expect ~100-1000 ps total runtime
on a single GPU. Use the largest supercell you can afford — NEMD
has strong finite-size effects (need L > phonon MFP ~300 nm for Si,
which is typically infeasible; extrapolation to 1/L → 0 is standard).
"""
import numpy as np
import time
import plotly.graph_objects as go
from ase import Atoms as AseAtoms
from ase.build.supercells import make_supercell
from ase.md.velocitydistribution import MaxwellBoltzmannDistribution
from ase.md.langevin import Langevin
from ase.md.verlet import VelocityVerlet
from ase import units

from alignn.ff.ff import AlignnAtomwiseCalculator
from jarvis.core.atoms import Atoms
from jarvis.db.figshare import get_jid_data


def make_elongated_cell(jid="JVASP-1002", nx=4, ny=4, nz=20, calc=None):
    """Build an elongated supercell: long axis = heat transport direction (z)."""
    d = get_jid_data(jid=jid, dataset="dft_3d")
    prim = Atoms.from_dict(d["atoms"]).get_conventional_atoms
    a = prim.ase_converter()
    a.calc = calc
    # quick relax
    from ase.filters import ExpCellFilter
    from ase.optimize import FIRE
    FIRE(ExpCellFilter(a), logfile=None).run(fmax=0.05, steps=100)
    # elongate along z
    sc = make_supercell(a, [[nx,0,0],[0,ny,0],[0,0,nz]])
    sc.calc = calc
    return sc


def equilibrate(atoms, T=300, steps=5000, dt_fs=1.0, friction=0.01):
    """Langevin thermalize to target T."""
    MaxwellBoltzmannDistribution(atoms, temperature_K=T)
    dyn = Langevin(atoms, dt_fs * units.fs, temperature_K=T,
                   friction=friction, logfile=None)
    t0 = time.time()
    dyn.run(steps)
    print(f"  equilibration: {steps} steps in {time.time()-t0:.1f}s")
    return atoms


def mueller_plathe_nemd(atoms, n_slabs=20, swap_interval=50, n_steps=100000,
                        dt_fs=1.0, profile_interval=100):
    """Reverse NEMD via kinetic-energy swaps.

    Divides the z-axis into n_slabs. The hottest atom in the 'cold' slab
    (slab 0) swaps velocities with the coldest atom in the 'hot' slab
    (slab n_slabs//2). This pumps kinetic energy 'uphill' — the atoms
    respond by building a real gradient from hot to cold.

    Flux J = ΔKE_cumulative / (2 · A · t · swap_interval · dt)
    (factor 2 because heat flows in both directions through the cell
    due to PBC — each swap drives flux to both sides.)
    """
    # NVE integrator during measurement
    dyn = VelocityVerlet(atoms, dt_fs * units.fs, logfile=None)

    cell = atoms.cell.array
    Lz = cell[2, 2]
    A_cross = np.linalg.norm(np.cross(cell[0], cell[1]))  # Å²
    slab_width = Lz / n_slabs
    cold_slab = 0
    hot_slab = n_slabs // 2

    masses = atoms.get_masses()                         # amu
    total_swapped_energy = 0.0                          # eV
    n_swaps_attempted = 0
    n_swaps_successful = 0

    # accumulators for T(z) profile
    T_profile_sum = np.zeros(n_slabs)
    T_profile_count = np.zeros(n_slabs)

    t0 = time.time()
    for step in range(n_steps):
        dyn.run(1)

        # --- periodic velocity swap ---
        if step % swap_interval == 0 and step > 0:
            n_swaps_attempted += 1
            z = atoms.get_positions()[:, 2] % Lz
            slab_id = np.floor(z / slab_width).astype(int) % n_slabs
            v = atoms.get_velocities()                  # Å/fs
            ke_atom = 0.5 * masses * np.sum(v**2, axis=1)  # in ASE internal units

            in_cold = np.where(slab_id == cold_slab)[0]
            in_hot  = np.where(slab_id == hot_slab)[0]
            if len(in_cold) == 0 or len(in_hot) == 0:
                continue

            # hottest in cold slab, coldest in hot slab
            i_hot_in_cold = in_cold[np.argmax(ke_atom[in_cold])]
            i_cold_in_hot = in_hot[np.argmin(ke_atom[in_hot])]

            ke_before = ke_atom[i_hot_in_cold] + ke_atom[i_cold_in_hot]

            # momentum-conserving velocity swap (equal masses trivial;
            # for unequal masses, scale to preserve KE on each atom)
            m1 = masses[i_hot_in_cold]
            m2 = masses[i_cold_in_hot]
            v1 = v[i_hot_in_cold].copy()
            v2 = v[i_cold_in_hot].copy()
            # scale by mass ratio to preserve momentum (Müller-Plathe convention)
            v[i_hot_in_cold] = v2 * np.sqrt(m2/m1)
            v[i_cold_in_hot] = v1 * np.sqrt(m1/m2)
            atoms.set_velocities(v)

            ke_after = (0.5*m1*np.sum(v[i_hot_in_cold]**2)
                        + 0.5*m2*np.sum(v[i_cold_in_hot]**2))
            delta_ke = ke_before - ke_after   # energy removed from cold slab
            # ke_atom values are in ASE units (amu·Å²/fs²); convert to eV
            # 1 amu·Å²/fs² = 1.0364e-4 eV... actually ASE uses its own unit
            # Use atoms.get_kinetic_energy() diff for safety:
            # Simpler: recompute explicitly in eV
            # ASE velocities are in Å/(ASE time unit); KE returned in eV.
            # We work directly with eV by using get_kinetic_energy before/after.
            total_swapped_energy += delta_ke
            n_swaps_successful += 1

        # --- accumulate T profile every profile_interval steps ---
        if step % profile_interval == 0:
            z = atoms.get_positions()[:, 2] % Lz
            slab_id = np.floor(z / slab_width).astype(int) % n_slabs
            v = atoms.get_velocities()
            ke_atom = 0.5 * masses * np.sum(v**2, axis=1)
            for s in range(n_slabs):
                mask = slab_id == s
                if mask.sum() > 0:
                    # T = (2/3) * <KE> / kB, in kelvin
                    # ke_atom here is in ASE units — use atoms method instead
                    kin_slab = np.sum(ke_atom[mask])
                    n_atoms_slab = mask.sum()
                    T_slab = (2.0/3.0) * kin_slab / (n_atoms_slab * units.kB)
                    T_profile_sum[s] += T_slab
                    T_profile_count[s] += 1

        if step % (n_steps // 20) == 0:
            elapsed = time.time() - t0
            print(f"  step {step}/{n_steps}  t={elapsed:.0f}s  "
                  f"swaps={n_swaps_successful}")

    # time-averaged profile
    T_profile = T_profile_sum / np.maximum(T_profile_count, 1)

    # flux: energy pumped per unit area per unit time
    # PBC means flux travels both ways → factor of 2
    total_time_fs = n_swaps_successful * swap_interval * dt_fs
    # convert A_cross from Å² to m², time from fs to s, energy from eV to J
    J = (total_swapped_energy * units._e) / (2 * A_cross * 1e-20 * total_time_fs * 1e-15)
    # J now in W/m²
    return T_profile, J, slab_width


def fit_gradient(T_profile, slab_width, n_slabs):
    """Fit dT/dz in the two linear regions (hot→cold and cold→hot).

    The profile is V-shaped because of PBC: temperature rises from cold
    to hot on one side and falls on the other. Fit both halves separately
    and average |dT/dz|.
    """
    z = (np.arange(n_slabs) + 0.5) * slab_width  # Å
    cold = 0
    hot = n_slabs // 2

    # left half: cold(0) → hot(n/2)
    idx_l = np.arange(cold + 1, hot)
    # right half: hot(n/2) → cold(n)  (wraps to cold via PBC)
    idx_r = np.arange(hot + 1, n_slabs - 1)

    slope_l = np.polyfit(z[idx_l], T_profile[idx_l], 1)[0]  # K/Å
    slope_r = np.polyfit(z[idx_r], T_profile[idx_r], 1)[0]

    dTdz_avg = 0.5 * (abs(slope_l) + abs(slope_r))  # K/Å
    dTdz_SI = dTdz_avg * 1e10                        # K/m
    return dTdz_SI, (z, T_profile, slope_l, slope_r)


# ================== run ==================
calc = AlignnAtomwiseCalculator(path="OutputDir", force_mult_batchsize=False,
                                device="cuda")

print("Building elongated Si supercell...")
# Conventional Si is 8 atoms; 4x4x20 = 1280 atoms. Lz ≈ 109 Å.
atoms = make_elongated_cell(jid="JVASP-1002", nx=3, ny=3, nz=15, calc=calc)
print(f"  N = {len(atoms)}, cell = {atoms.cell.lengths()}")

print("Equilibrating at 300 K (NVT)...")
atoms = equilibrate(atoms, T=300, steps=3000, dt_fs=1.0)

print("Running Müller-Plathe NEMD...")
T_profile, J, slab_w = mueller_plathe_nemd(
    atoms,
    n_slabs=20,
    swap_interval=50,        # swap every 50 fs
    n_steps=50000,           # 50 ps production
    dt_fs=1.0,
    profile_interval=100,
)

dTdz, fit_data = fit_gradient(T_profile, slab_w, n_slabs=20)
kappa = J / dTdz  # W/(m·K)

print(f"\n── NEMD κ summary ──")
print(f"  flux J = {J:.3e} W/m²")
print(f"  |dT/dz| = {dTdz:.3e} K/m = {dTdz*1e-9:.2f} K/nm")
print(f"  κ = {kappa:.2f} W/m·K")
print(f"  (finite-size; extrapolate 1/Lz → 0 for bulk)")

# --- plot T profile ---
z, Tp, sl, sr = fit_data
fig = go.Figure()
fig.add_trace(go.Scatter(x=z, y=Tp, mode="lines+markers", name="T(z)"))
fig.update_layout(
    template="plotly_white",
    xaxis_title="z (Å)",
    yaxis_title="T (K)",
    title=f"NEMD temperature profile — Si  κ={kappa:.1f} W/m·K",
    width=800, height=450,
)
fig.write_html("nemd_Tprofile.html")
fig.show()

STOP HERE

Interface design, gamma surface plot

In [ ]:
!pip install -q git+https://github.com/usnistgov/intermat.git@develop

In [ ]:
# Conventional cells
poscar_1= """JVASP1002
1.0
5.49363 0.0 0.0
-0.0 5.49363 0.0
0.0 0.0 5.49363
Si
8
direct
0.25 0.75 0.25 Si
0.0 0.0 0.5 Si
0.25 0.25 0.75 Si
0.0 0.5 0.0 Si
0.75 0.75 0.75 Si
0.5 0.0 0.0 Si
0.75 0.25 0.25 Si
0.5 0.5 0.5 Si
"""

poscar_2 = """JVASP-1174
1.0
5.73163 0.0 0.0
-0.0 5.73163 0.0
0.0 0.0 5.73163
Ga As
4 4
direct
0.0 0.0 0.0 Ga
0.0 0.5 0.5 Ga
0.5 0.0 0.5 Ga
0.5 0.5 0.0 Ga
0.25 0.75 0.75 As
0.25 0.25 0.25 As
0.75 0.75 0.25 As
0.75 0.25 0.75 As
"""


In [ ]:
from jarvis.db.jsonutils import loadjson
from jarvis.core.atoms import Atoms
from jarvis.io.vasp.inputs import Poscar
from intermat.generate import InterfaceCombi
import matplotlib.pyplot as plt

atoms1 = Poscar.from_string(poscar_1).atoms
atoms2 = Poscar.from_string(poscar_1).atoms #Poscar.from_string(poscar_2).atoms

x = InterfaceCombi(
    film_indices=[[1, 1, 0]],
    subs_indices=[[1, 1, 0]],
    vacuum_interface=2,
    film_mats=[atoms1],
    subs_mats=[atoms2],
    disp_intvl=0.0,
    from_conventional_structure_film=False,
    from_conventional_structure_subs=False,
    max_area=300,
    ltol=0.08,

)

structs = x.generate()
heterostructure_atoms=Atoms.from_dict(structs[0]['generated_interface'])
film_atoms=Atoms.from_dict(structs[0]['film_surf'])
surf_atoms=Atoms.from_dict(structs[0]['subs_surf'])


In [ ]:
print(heterostructure_atoms)

Note it can be a high energy if slabs are randomly put together, need xy-plane scan

In [ ]:
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt
%matplotlib inline
plot_atoms(heterostructure_atoms.ase_converter(),  radii=0.3, rotation=('90x,90y,0z'))
plt.axis('off')


In [ ]:
from jarvis.db.jsonutils import loadjson
from jarvis.core.atoms import Atoms
from jarvis.io.vasp.inputs import Poscar
from intermat.generate import InterfaceCombi
import matplotlib.pyplot as plt

atoms1 = Poscar.from_string(poscar_1).atoms
atoms2 = Poscar.from_string(poscar_1).atoms

# disp_intvl = 0.1 for xy-plane scan
x = InterfaceCombi(
    film_indices=[[1, 1, 0]],
    subs_indices=[[1, 1, 0]],
    vacuum_interface=2,
    film_mats=[atoms1],
    subs_mats=[atoms2],
    disp_intvl=0.1,
    from_conventional_structure_film=False,
    from_conventional_structure_subs=False,
    max_area=300,
    ltol=0.08,

)

structs = x.generate()
extra_params={}
extra_params["alignn_params"] ={}
extra_params["alignn_params"]["model_path"] = dir_name
wads = x.calculate_wad(method='alignn_ff',extra_params=extra_params)
# heterostructure_atoms=Atoms.from_dict(structs[0]['generated_interface'])
# film_atoms=Atoms.from_dict(structs[0]['film_surf'])
# surf_atoms=Atoms.from_dict(structs[0]['subs_surf'])


In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
X = x.X
Y = x.Y
wads = np.array(wads).reshape(len(X), len(Y))
wads = np.array(x.wads["wads"])
index = np.argmin(wads)
combined_atoms = Atoms.from_dict(
    x.generated_interfaces[index]["generated_interface"]
)
wads = np.array(wads).reshape(len(X), len(Y))
plt.contourf(wads,cmap='plasma')


In [ ]:
print(combined_atoms)

In [ ]:
# After xy plane scan we can see the diamond structure pattern continues,
# which is desirable for getting low energy structure
from ase.visualize.plot import plot_atoms
import matplotlib.pyplot as plt
%matplotlib inline
plot_atoms(combined_atoms.ase_converter(),  radii=0.3, rotation=('90x,90y,0z'))
plt.axis('off')


In [ ]:
import plotly.graph_objects as go

fig = go.Figure(data=[go.Surface(z=wads)])

# fig = go.Figure(data=[go.Contour(z=res['wads'])])
#fig.write_html("file2.html")
fig.show()

General optimization from POSCAR

In [ ]:
from jarvis.io.vasp.inputs import Poscar
from ase.constraints import ExpCellFilter
from ase.optimize.fire import FIRE

def general_relaxer(atoms="", calculator="", fmax=0.05, steps=150,relax=True):
    ase_atoms = atoms.ase_converter()
    ase_atoms.calc = calculator
    if not relax:
         return ase_atoms.get_potential_energy()

    ase_atoms = ExpCellFilter(ase_atoms)

    dyn = FIRE(ase_atoms)
    dyn.run(fmax=fmax, steps=steps)
    return ase_to_atoms(ase_atoms.atoms)

pos="""Si2
1.0
3.3641499856336465 -2.5027128e-09 1.94229273881412
1.121382991333525 3.1717517190189715 1.9422927388141193
-2.5909987e-09 -1.8321133e-09 3.884586486670313
Si
2
Cartesian
3.92483875 2.77528125 6.7980237500000005
0.56069125 0.39646875 0.9711462500000001
"""
s = Poscar.from_string(pos).atoms.get_conventional_atoms
opt = general_relaxer(atoms=s,calculator=calc)
print(s)
print()
print(opt)

Scaling

In [ ]:
from ase.build.supercells import make_supercell
import numpy as np
import time
from alignn.ff.ff import AlignnAtomwiseCalculator, default_path, ev_curve
import pandas as pd
import numpy as np
import zipfile
import json
import os
import glob
import matplotlib.pyplot as plt
from ase.stress import full_3x3_to_voigt_6_stress, voigt_6_to_full_3x3_stress
from ase import Atoms,Atom
A = 2.7223*np.array([[1,1,0],[1,0,1],[0,1,1]])
model_path = "OutputDir"
dir_name = model_path
calc = AlignnAtomwiseCalculator(
    path=model_path,
    force_mult_natoms=False,
    force_multiplier=my_config['batch_size'],
    stress_wt=0.3,
)
atoms_si = Atoms([Atom('Si', (0, 0, 0)), Atom('Si', (0.25, 0.25, 0.25))], cell = A, pbc=True)
atoms_si.calc = calc
en = atoms_si.get_potential_energy()
numbers = np.arange(1,9,1)
times = []
natoms=[]
for i in numbers:

  sc=make_supercell(atoms_si, [[i,0,0],[0,i,0],[0,0,i]])
  t1=time.time()
  sc.calc = calc

  en = sc.get_potential_energy()/len(sc)
  t2=time.time()

  times.append(t2-t1)
  natoms.append(len(sc))
  print(i,len(sc),en,t2-t1)

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
plt.plot(natoms,times,'o-')
plt.xlabel('Number of atoms')
plt.ylabel('Time (s)')
plt.show()

In [ ]:
!conda env export > environment.yml

In [ ]:
!pip freeze

In [ ]:
!conda env export